# Agenthon Track 2 — VStack Short V2 검증

공식 22개 short-horizon practice card를 같은 5개 seed로 평가한 재현 가능한 분석 노트북입니다. 공식 정규화 점수는 **낮을수록 좋고**, 카드별 동일 가중 평균을 사용합니다. seed는 Monte Carlo 오차 확인용이며 독립 표본 수는 110이 아니라 22개 카드입니다.

In [1]:
from analyze_final import load_checked, overall_table, seed_table, segment_table, card_table, robustness
candidates, merged = load_checked()
overall = overall_table(merged)
seeds = seed_table(merged)
overall.round(6)

,model,score,improvement_vs_m0_pct,improvement_vs_submitted_pct
0,VStack Short V2,1.061779,7.043976,8.491194
1,VStack Short Safe,1.068116,6.489232,7.945086
2,Gaussian M0,1.142238,0.000000,1.556884
3,Submitted VStack Stable,1.160303,-1.581506,0.000000


In [2]:
seeds.round(6)

,candidate,seed,cards,score,m0_score,current_vstack_score,improvement_vs_m0_pct,improvement_vs_submitted_pct
0,vstack_short_safe,0,22,1.057505,1.142238,1.153014,7.418213,8.283479
1,vstack_short_safe,1,22,1.072789,1.142238,1.155555,6.080096,7.162456
2,vstack_short_safe,42,22,1.072221,1.142238,1.167148,6.129830,8.133245
3,vstack_short_safe,2026,22,1.064108,1.142238,1.164304,6.840135,8.605725
4,vstack_short_safe,314159,22,1.073956,1.142238,1.161492,5.977886,7.536479
5,vstack_short_v2,0,22,1.050111,1.142238,1.153014,8.065495,8.924712
6,vstack_short_v2,1,22,1.067175,1.142238,1.155555,6.571570,7.648266
7,vstack_short_v2,42,22,1.063824,1.142238,1.167148,6.864941,8.852666
8,vstack_short_v2,2026,22,1.058770,1.142238,1.164304,7.307399,9.064133
9,vstack_short_v2,314159,22,1.069015,1.142238,1.161492,6.410477,7.961899


In [3]:
groups = segment_table(merged, 'group')
families = segment_table(merged, 'family')
cards = card_table(merged)
groups.round(6), families.round(6)

(           candidate      group  cards  observations     score  m0_score  \
 0    vstack_short_v2      EM FX     11            55  0.979277  0.983261   
 1  vstack_short_safe      EM FX     11            55  0.989606  0.983261   
 2    vstack_short_v2     G10 FX      3            15  1.479896  1.539385   
 3  vstack_short_safe     G10 FX      3            15  1.488492  1.539385   
 4  vstack_short_safe  UST bills      8            40  1.018426  1.211902   
 5    vstack_short_v2  UST bills      8            40  1.018426  1.211902   
 
    current_vstack_score  improvement_vs_m0_pct  improvement_vs_submitted_pct  
 0              1.108377               0.405176                     11.647670  
 1              1.108377              -0.645261                     10.715809  
 2              1.569492               3.864473                      5.708627  
 3              1.569492               3.306046                      5.160912  
 4              1.078254              15.964650            

In [4]:
robustness(cards)

{'vstack_short_safe': {'cards': 22,
  'unique_asof_dates': 22,
  'card_mean_delta_vs_m0': -0.0741224837527039,
  'card_cluster_bootstrap_95pct_delta_vs_m0': [-0.1668797087569964,
   0.0012332170329800553],
  'card_mean_delta_vs_submitted': -0.092187050979375,
  'card_cluster_bootstrap_95pct_delta_vs_submitted': [-0.23327933300186288,
   0.00421988151659057],
  'leave_one_card_delta_vs_m0_range': [-0.08264513876862489,
   -0.04763095911370072],
  'leave_one_card_delta_vs_submitted_range': [-0.104361119208382,
   -0.03277385833294757],
  'leave_one_asof_delta_vs_m0_range': [-0.08264513876862489,
   -0.04763095911370072],
  'leave_one_asof_delta_vs_submitted_range': [-0.104361119208382,
   -0.03277385833294757],
  'card_win_rate_vs_m0': 0.5909090909090909,
  'card_win_rate_vs_submitted': 0.5},
 'vstack_short_v2': {'cards': 22,
  'unique_asof_dates': 22,
  'card_mean_delta_vs_m0': -0.08045898490030196,
  'card_cluster_bootstrap_95pct_delta_vs_m0': [-0.17246214554126577,
   -0.0056668865500

## 핵심 비교

V2는 국채 F1/F2에서 기존 수치 엔진을 유지하고, 나머지 일별 레벨 카드에는 cutoff-local Gaussian random walk를 사용합니다. FX F3만 연속 과거경로를 사용해 공동분포를 보존합니다. 카드 ID, 실현값 또는 정답 조회 규칙은 런타임 모델에 포함하지 않습니다.

In [5]:
from IPython.display import Markdown, display
display(Markdown('![Five-seed score improvement versus Gaussian M0](score_improvement_vs_m0.png)'))

![Five-seed score improvement versus Gaussian M0](score_improvement_vs_m0.png)

## 해석상 제한

- 공개 22개 카드는 historical practice set이며 Final의 독립 holdout이 아닙니다.
- Final은 같은 카드 형태이지만 하나의 더 늦은 as-of 날짜를 사용하므로 시점 전이는 불확실합니다.
- 로컬 환경에서는 House API를 사용할 수 없어 numerical fallback만 측정했습니다.
- 따라서 7.04%는 공개 practice set의 추정치이며 실제 Final 점수를 보장하지 않습니다.